# Phần 2: ablation tách nhóm đặc trưng và đánh giá độ tin cậy của LSTM trên FreshRetailNet-50K

Notebook này dùng lại quy trình của phần 1 (FreshRetailNet_LSTM.ipynb): cùng tập con 1,000 chuỗi, cùng quy tắc hết hàng, cùng kiến trúc LSTM.

| Thí nghiệm | Mô hình | Đầu vào |
|---|---|---|
| E1 | Seasonal Naive 7 ngày (mốc tham chiếu) | y(t−7) |
| E2 | LSTM | doanh số, lịch, khuyến mãi, thời tiết |
| E3 | LSTM | bỏ khuyến mãi và thời tiết |
| E3a | LSTM | bỏ khuyến mãi (discount, activity_flag), giữ thời tiết và lịch |
| E3b | LSTM | bỏ thời tiết, giữ khuyến mãi và lịch |
| E4 | LSTM | như E2, không loại ngày hết hàng khỏi hàm mất mát |

RQ3: đóng góp riêng của khuyến mãi và của thời tiết; mức quá khớp của E2 so với E3b.
RQ5: kết quả có ổn định qua 5 seed × 5 tuần kiểm tra trượt không, và chênh lệch có ý nghĩa thống kê không.

In [ ]:
# Colab: bỏ comment nếu thiếu thư viện
# !pip install -q huggingface_hub scipy

## 0. Cấu hình

In [ ]:
N_SERIES = 1000
SUBSET_SEED = 42           # seed chọn tập con, giống phần 1
SEEDS = [42, 43, 44, 45, 46]
K_WINDOWS = 5              # số tuần kiểm tra trượt (tuần cuối cùng là tập eval)
MODELS = ["E2", "E3", "E3a", "E3b", "E4"]
STOCKOUT_MAX_HOURS = 8
CAP_QUANTILE = 0.99
WINDOW = 28
HORIZON = 7
HIDDEN = 64
EPOCHS_MAX = 30
PATIENCE = 4
BATCH_SIZE = 256
LR = 1e-3
N_WORKERS = 6              # số tiến trình song song (mỗi tiến trình 1 luồng torch), 1 = chạy tuần tự
ALPHA = 0.05
HF_REPO = "Dingdong-Inc/FreshRetailNet-50K"
HF_REVISION = "08c1fab7f9257bc73679d415d65d644165d351d4"
DATA_SOURCE = "hf"
LOCAL_DATA_DIR = "data"
OUTPUT_DIR = "outputs"
FIG_DIR = "figures"
PART1_RESULTS = "outputs/results.csv"   # kết quả phần 1, để đối chiếu

## 1. Thư viện

In [ ]:
import os, time, json, random, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.compute as pc
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy import stats

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
T0 = time.time(); TIMES = {}
def mark(name):
    TIMES[name] = round(time.time() - T0, 1); print(f"[{name}] {TIMES[name]} s kể từ đầu")
import scipy
print("numpy", np.__version__, "| pandas", pd.__version__, "| torch", torch.__version__, "| scipy", scipy.__version__, "| CPU", os.cpu_count())

## 2. Tải dữ liệu và chọn tập con (như phần 1)

Để tiết kiệm bộ nhớ, chỉ đọc 5 cột để tính tầng, rồi lọc bảng Arrow theo các chuỗi được chọn trước khi chuyển sang pandas.

In [ ]:
DAILY_COLS = ["city_id", "store_id", "management_group_id", "product_id", "dt", "sale_amount",
              "stock_hour6_22_cnt", "discount", "holiday_flag", "activity_flag",
              "precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
def locate_files():
    if DATA_SOURCE == "hf":
        try:
            from huggingface_hub import hf_hub_download
            return {s: hf_hub_download(HF_REPO, f"data/{s}.parquet", repo_type="dataset", revision=HF_REVISION)
                    for s in ("train", "eval")}, "huggingface"
        except Exception as e:
            print("Tải từ Hugging Face lỗi, dùng bản cục bộ:", repr(e)[:150])
    files = {s: os.path.join(LOCAL_DATA_DIR, f"{s}.parquet") for s in ("train", "eval")}
    assert all(os.path.exists(p) for p in files.values()), f"Không thấy dữ liệu: {files}"
    return files, "local"
FILES, SRC = locate_files()
small = pq.read_table(FILES["train"], columns=["city_id", "store_id", "product_id", "management_group_id", "stock_hour6_22_cnt"]).to_pandas()
print("Nguồn:", SRC, "| bảng train (5 cột):", small.shape)
ser = (small.groupby(["store_id", "product_id"])
       .agg(mg=("management_group_id", "first"), city=("city_id", "first"),
            so_rate=("stock_hour6_22_cnt", lambda s: s.mean() / 16)).reset_index())
del small
ser["so_level"] = pd.qcut(ser["so_rate"].rank(method="first"), 3, labels=["thấp", "vừa", "cao"])
ser["stratum"] = ser["mg"].astype(str) + "_" + ser["so_level"].astype(str)
rng = np.random.default_rng(SUBSET_SEED)
alloc = (ser["stratum"].value_counts() / len(ser) * N_SERIES)
n_take = np.floor(alloc).astype(int)
n_take[(alloc - np.floor(alloc)).sort_values(ascending=False).index[:N_SERIES - n_take.sum()]] += 1
chosen = pd.concat([g.iloc[rng.permutation(len(g))[:n_take[s]]] for s, g in ser.groupby("stratum")])
chosen = chosen.sort_values(["store_id", "product_id"]).reset_index(drop=True)
print("Trước: số chuỗi =", len(ser), "| sau: số chuỗi =", len(chosen))
key = chosen.store_id.astype(np.int64) * 100000 + chosen.product_id.astype(np.int64)
parts = []
for s in ("train", "eval"):
    t = pq.read_table(FILES[s], columns=DAILY_COLS)
    k = pc.add(pc.multiply(pc.cast(t["store_id"], "int64"), 100000), pc.cast(t["product_id"], "int64"))
    t = t.filter(pc.is_in(k, value_set=pa.array(key.values)))
    print(f"{s}: {pq.read_metadata(FILES[s]).num_rows:,} dòng -> {t.num_rows:,} dòng sau khi lọc")
    parts.append(t.to_pandas())
df = pd.concat(parts).sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True); del parts
assert (df.groupby(["store_id", "product_id"]).size() == 97).all()
df["dt"] = pd.to_datetime(df["dt"]); days = np.sort(df["dt"].unique())
df["day"] = np.searchsorted(days, df["dt"].values); df["dow"] = df["dt"].dt.dayofweek
print("Bảng theo ngày:", df.shape, "| giá trị thiếu:", int(df.isna().sum().sum()))
display(df.head(3))
mark("load_subset")

## 3. Nhắc lại EDA cho thời tiết

Tương quan hạng Spearman giữa thời tiết và doanh số trên 83 ngày huấn luyện của phần 1 (ngày 0–82). Dùng để diễn giải RQ3.

In [ ]:
tr = df[df.day <= 82]
sp = {c: tr[c].rank().corr(tr.sale_amount.rank()) for c in ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]}
for c, v in sp.items(): print(f"Spearman({c}, sale_amount) = {v:.3f}")
sp_promo = {c: tr[c].rank().corr(tr.sale_amount.rank()) for c in ["discount", "activity_flag"]}
for c, v in sp_promo.items(): print(f"Spearman({c}, sale_amount) = {v:.3f}")
del tr

## 4. Mảng theo chuỗi × ngày

Chuyển bảng dài sang mảng N × 97 cho từng biến. Các bước xử lý có dùng thống kê (sửa discount, chặn p99, chuẩn hóa) được làm lại cho từng cửa sổ, chỉ với ngày huấn luyện của cửa sổ đó.

In [ ]:
df = df.sort_values(["store_id", "product_id", "day"]).reset_index(drop=True)
N = df.groupby(["store_id", "product_id"]).ngroups; D = 97
def A(c, dtype=np.float64): return df[c].to_numpy(dtype).reshape(N, D)
WEATHER = ["precpt", "avg_temperature", "avg_humidity", "avg_wind_level"]
Y64 = A("sale_amount"); Y = Y64.astype(np.float32)
DISC_RAW = A("discount"); ACT = A("activity_flag"); HOL = A("holiday_flag"); DOW = A("dow", np.int64)
WX = np.stack([A(c) for c in WEATHER], -1)
PROD = df["product_id"].to_numpy().reshape(N, D)[:, 0]
CNT = A("stock_hour6_22_cnt")
KEEP = (CNT <= STOCKOUT_MAX_HOURS).astype(np.float32)   # quan sát được sau mỗi ngày, không rò rỉ
print("Y:", Y.shape, "| WX:", WX.shape, "| discount = 0:", int((DISC_RAW == 0).sum()), "| ngày bị loại (cnt > 8):", int((KEEP == 0).sum()))

## 5. Thiết kế cửa sổ trượt (walk-forward)

Với tuần kiểm tra bắt đầu ở ngày s: huấn luyện trên mục tiêu đến ngày s − 8, validation là tuần s − 7 … s − 1 (early stopping), sau đó huấn luyện lại đến ngày s − 1 với số epoch tốt nhất và dự báo tuần s … s + 6 từ mốc s − 1. Cửa sổ cuối (s = 90) trùng với phần 1.

In [ ]:
STARTS = [90 - 7 * k for k in range(K_WINDOWS)][::-1]
WIN = []
for w, s in enumerate(STARTS):
    te = s - 8
    WIN.append({"w": w + 1, "test_start": s, "train_end": te, "train_origins": f"{WINDOW-1}–{te-HORIZON}",
                "n_train_origins": te - HORIZON - (WINDOW - 1) + 1, "val_origin": te, "test_origin": s - 1,
                "train": f"{pd.Timestamp(days[0]):%d/%m}–{pd.Timestamp(days[te]):%d/%m}",
                "val": f"{pd.Timestamp(days[s-7]):%d/%m}–{pd.Timestamp(days[s-1]):%d/%m}",
                "test": f"{pd.Timestamp(days[s]):%d/%m}–{pd.Timestamp(days[s+6]):%d/%m}"})
WIN_DF = pd.DataFrame(WIN).set_index("w"); display(WIN_DF)
WIN_DF.to_csv(f"{OUTPUT_DIR}/p2_windows.csv")
for r in WIN: assert r["val_origin"] + HORIZON < r["test_start"] and r["train_end"] < r["test_start"] - 7

## 6. Tiền xử lý theo từng cửa sổ

Với mỗi cửa sổ: điền discount = 0 bằng trung vị sản phẩm × activity_flag; chặn doanh số đầu vào ở p99 của chuỗi; chia cho trung bình chuỗi; z-score thời tiết. Mọi thống kê chỉ dùng ngày ≤ train_end.

In [ ]:
DOW1H = np.eye(7, dtype=np.float64)[DOW]            # N × 97 × 7
def prepare(te):
    trm = np.zeros(D, bool); trm[:te + 1] = True
    disc = DISC_RAW.copy()
    long = pd.DataFrame({"p": np.repeat(PROD, te + 1), "a": ACT[:, :te + 1].ravel(), "d": DISC_RAW[:, :te + 1].ravel()})
    long = long[long.d > 0]
    med_pa = long.groupby(["p", "a"])["d"].median().to_dict(); med_a = long.groupby("a")["d"].median().to_dict()
    for i, t in zip(*np.where(disc == 0)):
        disc[i, t] = med_pa.get((PROD[i], ACT[i, t]), med_a[ACT[i, t]])
    disc = np.minimum(disc, 1.0)
    p99 = np.quantile(Y64[:, trm], CAP_QUANTILE, axis=1)
    sale_in = np.minimum(Y64, p99[:, None])
    scale = np.clip(sale_in[:, trm].mean(1), 0.01, None)
    mu = WX[:, trm].reshape(-1, 4).mean(0); sd = WX[:, trm].reshape(-1, 4).std(0, ddof=1)
    wz = (WX - mu) / sd
    feats = {"sale_n": (sale_in / scale[:, None]), "dow": DOW1H, "hol": HOL[..., None], "promo": np.stack([disc, ACT], -1), "wx": wz}
    feats = {k: (v[..., None] if v.ndim == 2 else v).astype(np.float32) for k, v in feats.items()}
    info = {"train_end": te, "discount_filled": int((DISC_RAW == 0).sum()), "capped_train": int((Y64[:, trm] > p99[:, None]).sum()),
            "capped_all": int((Y64 > p99[:, None]).sum()), "scale_mean": float(scale.mean()), "wx_mu": mu.round(3).tolist(), "wx_sd": sd.round(3).tolist()}
    return feats, scale.astype(np.float32), info
te_last = WIN[-1]["train_end"]
print("Trước (cửa sổ cuối): doanh số gốc", pd.Series(Y64.ravel()).describe().round(3).to_dict())
F_last, S_last, I_last = prepare(te_last)
print("Sau: sale_n", pd.Series(F_last["sale_n"].ravel()).describe().round(3).to_dict())
print("Sau: discount còn bằng 0:", int((F_last["promo"][..., 0] == 0).sum()), "| thời tiết z (ngày huấn luyện) trung bình:",
      F_last["wx"][:, :te_last + 1].reshape(-1, 4).mean(0).round(3))
PREP = {}
for r in WIN:
    feats, scale, info = prepare(r["train_end"]); PREP[r["w"]] = (feats, scale); r.update(info)
PREP_DF = pd.DataFrame([{k: r[k] for k in ["w", "train_end", "discount_filled", "capped_train", "capped_all", "scale_mean"]} for r in WIN]).set_index("w")
display(PREP_DF)
mark("prepare")

## 7. Bộ đặc trưng của từng thí nghiệm

In [ ]:
FEATSETS = {"E2": ["promo", "wx"], "E3": [], "E3a": ["wx"], "E3b": ["promo"], "E4": ["promo", "wx"]}
def build(feats, extra):
    P = np.concatenate([feats["sale_n"], feats["dow"], feats["hol"]] + [feats[g] for g in extra], -1)
    F = np.concatenate([feats["dow"], feats["hol"]] + [feats[g] for g in extra], -1)
    return P, F
rows = []
for m in MODELS:
    P, F = build(PREP[K_WINDOWS][0], FEATSETS[m])
    rows.append({"thí nghiệm": m, "nhóm thêm": "+".join(FEATSETS[m]) or "(chỉ doanh số, lịch)", "biến quá khứ": P.shape[-1], "biến biết trước": F.shape[-1],
                 "loại ngày hết hàng": m != "E4"})
display(pd.DataFrame(rows).set_index("thí nghiệm"))

## 8. Cửa sổ mẫu, mô hình và huấn luyện

Giống phần 1: LSTM một lớp 64 đơn vị, nhánh biến biết trước, MSE có mặt nạ, Adam, early stopping theo WAPE validation, huấn luyện lại với số epoch tốt nhất.

In [ ]:
def windows(P, F, YN, keep, origins):
    i = np.repeat(np.arange(N), len(origins)); t = np.tile(origins, N)
    past = np.stack([P[i, t - WINDOW + 1 + k] for k in range(WINDOW)], 1)
    fut = np.stack([F[i, t + 1 + k] for k in range(HORIZON)], 1)
    y = np.stack([YN[i, t + 1 + k] for k in range(HORIZON)], 1)
    m = np.stack([keep[i, t + 1 + k] for k in range(HORIZON)], 1)
    return past, fut, y, m, i, t

class LSTMForecaster(nn.Module):
    def __init__(self, n_past, n_fut):
        super().__init__()
        self.lstm = nn.LSTM(n_past, HIDDEN, batch_first=True)
        self.head = nn.Sequential(nn.Linear(HIDDEN + HORIZON * n_fut, 128), nn.ReLU(), nn.Linear(128, HORIZON))
    def forward(self, xp, xf):
        _, (h, _) = self.lstm(xp)
        return self.head(torch.cat([h[-1], xf.flatten(1)], 1))

def wape(pred, true, mask): return float(np.abs(pred - true)[mask].sum() / true[mask].sum())

def predict_origins(model, P, F, YN, keep, scale, origins):
    xp, xf, _, _, i, t = windows(P, F, YN, keep, origins)
    model.eval()
    with torch.no_grad():
        out = torch.cat([model(torch.from_numpy(xp[b:b+8192]), torch.from_numpy(xf[b:b+8192])) for b in range(0, len(xp), 8192)]).numpy()
    return np.clip(out * scale[i][:, None], 0, None), i, t

def fit(P, F, YN, keep, scale, origins, epochs, seed, val_origin=None):
    seed_everything(seed)
    xp, xf, y, m, _, _ = windows(P, F, YN, keep, origins)
    model = LSTMForecaster(xp.shape[2], xf.shape[2]); opt = torch.optim.Adam(model.parameters(), lr=LR)
    T = [torch.from_numpy(a) for a in (xp, xf, y, m)]
    g = torch.Generator().manual_seed(seed)
    best, best_ep, bad = np.inf, 0, 0
    for ep in range(1, epochs + 1):
        model.train(); perm = torch.randperm(len(xp), generator=g)
        for b in range(0, len(xp), BATCH_SIZE):
            j = perm[b:b + BATCH_SIZE]
            out = model(T[0][j], T[1][j])
            loss = ((out - T[2][j]) ** 2 * T[3][j]).sum() / T[3][j].sum().clamp(min=1)
            opt.zero_grad(); loss.backward(); opt.step()
        if val_origin is not None:
            pv, _, _ = predict_origins(model, P, F, YN, keep, scale, np.array([val_origin]))
            v = wape(pv, Y[:, val_origin + 1:val_origin + 8], KEEP[:, val_origin + 1:val_origin + 8] > 0)
            if v < best - 1e-5: best, best_ep, bad = v, ep, 0
            else: bad += 1
            if bad >= PATIENCE: break
    return model, best_ep, best

def run_task(task):
    m, seed, w = task
    t0 = time.time(); torch.set_num_threads(1)
    r = WIN[w - 1]; s, te = r["test_start"], r["train_end"]
    feats, scale = PREP[w]
    P, F = build(feats, FEATSETS[m]); YN = Y / scale[:, None]
    keep = np.ones_like(KEEP) if m == "E4" else KEEP
    tr_orig = np.arange(WINDOW - 1, te - HORIZON + 1); full_orig = np.arange(WINDOW - 1, s - HORIZON)
    _, best_ep, best_val = fit(P, F, YN, keep, scale, tr_orig, EPOCHS_MAX, seed, val_origin=te)
    model, _, _ = fit(P, F, YN, keep, scale, full_orig, best_ep, seed)
    pred, _, _ = predict_origins(model, P, F, YN, keep, scale, np.array([s - 1]))
    ptr, i, t = predict_origins(model, P, F, YN, keep, scale, full_orig)       # sai số trên dữ liệu huấn luyện (trong mẫu)
    ytr = np.stack([Y[i, t + 1 + k] for k in range(HORIZON)], 1); mtr = np.stack([KEEP[i, t + 1 + k] for k in range(HORIZON)], 1) > 0
    return {"model": m, "seed": seed, "w": w, "best_epoch": best_ep, "val_wape": best_val, "train_wape": wape(ptr, ytr, mtr),
            "pred": pred.astype(np.float32), "sec": round(time.time() - t0, 1)}
print("Số tác vụ LSTM:", len(MODELS) * len(SEEDS) * K_WINDOWS)

## 9. Kiểm tra tính tất định

Chạy E2, seed 42, cửa sổ cuối hai lần trong cùng tiến trình và so sánh dự báo. Cửa sổ cuối có cùng dữ liệu và seed như E2 của phần 1, nên WAPE phải trùng với phần 1.

In [ ]:
YT_last = Y[:, 90:97]; MT_last = KEEP[:, 90:97] > 0
a = run_task(("E2", 42, K_WINDOWS)); b = run_task(("E2", 42, K_WINDOWS))
print("Hai lần chạy giống hệt:", bool(np.array_equal(a["pred"], b["pred"])), "| best_epoch", a["best_epoch"], b["best_epoch"], "| thời gian", a["sec"], "s")
w2 = wape(a["pred"], YT_last, MT_last); print("WAPE E2 (seed 42, cửa sổ cuối): %.4f" % w2)
if os.path.exists(PART1_RESULTS):
    p1 = pd.read_csv(PART1_RESULTS, index_col=0).loc["E2", "WAPE"]; print("WAPE E2 của phần 1: %.4f | trùng: %s" % (p1, abs(p1 - w2) < 1e-6))
DET = {"identical": bool(np.array_equal(a["pred"], b["pred"])), "wape_e2_seed42_last": w2}
mark("determinism")

## 10. Chạy toàn bộ: 5 thí nghiệm LSTM × 5 seed × 5 cửa sổ

Chạy song song bằng nhiều tiến trình (fork), mỗi tiến trình 1 luồng torch. Mỗi tác vụ tự đặt seed nên kết quả không phụ thuộc thứ tự chạy.

In [ ]:
TASKS = [(m, s, w) for w in range(1, K_WINDOWS + 1) for m in MODELS for s in SEEDS]
t_run = time.time(); OUT = []
if N_WORKERS > 1:
    with ProcessPoolExecutor(N_WORKERS, mp_context=mp.get_context("fork")) as ex:
        for k, r in enumerate(ex.map(run_task, TASKS, chunksize=1)):
            OUT.append(r)
            if (k + 1) % 25 == 0: print(f"  {k+1}/{len(TASKS)} tác vụ, {time.time()-t_run:.0f} s")
else:
    OUT = [run_task(t) for t in TASKS]
same = [r for r in OUT if (r["model"], r["seed"], r["w"]) == ("E2", 42, K_WINDOWS)][0]
print("Tác vụ (E2, 42, cửa sổ cuối) trong pool giống lần chạy tuần tự:", bool(np.array_equal(same["pred"], a["pred"])))
RUNS = pd.DataFrame([{k: v for k, v in r.items() if k != "pred"} for r in OUT])
print("Tổng thời gian huấn luyện (cộng các tác vụ): %.0f s | thời gian thực: %.0f s" % (RUNS.sec.sum(), time.time() - t_run))
display(RUNS.groupby("model")[["best_epoch", "sec"]].describe().round(1))
mark("train_all")

## 11. Dự báo và chỉ số theo từng lần chạy

E1 không học nên giống nhau giữa các seed; để ghép cặp, E1 được lặp lại cho mỗi seed. Đánh giá chính trên chuỗi – ngày có tối đa 8 giờ hết hàng.

In [ ]:
PRED = {}
for r in OUT: PRED[(r["model"], r["seed"], r["w"])] = r["pred"]
for w, rw in enumerate(WIN, 1):
    s = rw["test_start"]
    for sd in SEEDS: PRED[("E1", sd, w)] = Y[:, s - 7:s].copy()
ALL_MODELS = ["E1"] + MODELS
def yt(w): s = WIN[w - 1]["test_start"]; return Y[:, s:s + 7], KEEP[:, s:s + 7] > 0, ACT[:, s:s + 7] > 0
rows = []
for (m, sd, w), p in PRED.items():
    y, mk, pr = yt(w); e = (p - y)[mk]
    rows.append({"model": m, "seed": sd, "w": w, "WAPE": np.abs(e).sum() / y[mk].sum(), "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()),
                 "WAPE_promo": wape(p, y, mk & pr), "WAPE_nonpromo": wape(p, y, mk & ~pr), "n_eval": int(mk.sum())})
MET = pd.DataFrame(rows).sort_values(["model", "w", "seed"]).reset_index(drop=True)
MET.to_csv(f"{OUTPUT_DIR}/p2_metrics_runs.csv", index=False)
print("Số chuỗi – ngày được đánh giá theo cửa sổ:", MET.groupby("w").n_eval.first().to_dict())
display(MET.head(8).round(4))

## 12. RQ5: trung bình ± độ lệch chuẩn qua seed × cửa sổ

In [ ]:
def mstd(g, c): return g[c].mean(), g[c].std(ddof=1)
SUM = MET.groupby("model").agg(**{f"{c}_{f}": (c, f) for c in ["WAPE", "MAE", "RMSE"] for f in ["mean", "std"]},
                               n=("WAPE", "size")).reindex(ALL_MODELS)
SUM["WAPE_giam_vs_E1_%"] = (1 - SUM.WAPE_mean / SUM.loc["E1", "WAPE_mean"]) * 100
display(SUM.round(4)); SUM.to_csv(f"{OUTPUT_DIR}/p2_summary.csv")
BYW = MET.groupby(["w", "model"]).WAPE.mean().unstack().reindex(columns=ALL_MODELS)
BYW.index = [f"{w}: {WIN[w-1]['test']}" for w in BYW.index]
print("WAPE trung bình 5 seed theo cửa sổ:"); display(BYW.round(4)); BYW.to_csv(f"{OUTPUT_DIR}/p2_wape_by_window.csv")
SEEDSTD = MET[MET.model != "E1"].groupby(["model", "w"]).WAPE.std(ddof=1).groupby("model").mean()
print("Độ lệch chuẩn WAPE giữa các seed (trung bình qua cửa sổ):"); print(SEEDSTD.round(4).to_string())

## 13. RQ3: ablation tách nhóm khuyến mãi và thời tiết

In [ ]:
ABL = MET[MET.model.isin(["E2", "E3", "E3a", "E3b"])].groupby("model").agg(
    WAPE_mean=("WAPE", "mean"), WAPE_std=("WAPE", "std"), MAE_mean=("MAE", "mean"), RMSE_mean=("RMSE", "mean"),
    WAPE_promo=("WAPE_promo", "mean"), WAPE_nonpromo=("WAPE_nonpromo", "mean")).reindex(["E2", "E3a", "E3b", "E3"])
ABL["ΔWAPE_vs_E2"] = ABL.WAPE_mean - ABL.loc["E2", "WAPE_mean"]
ABL["ΔWAPE_vs_E3"] = ABL.WAPE_mean - ABL.loc["E3", "WAPE_mean"]
display(ABL.round(4)); ABL.to_csv(f"{OUTPUT_DIR}/p2_ablation.csv")
piv = MET.pivot_table(index=["seed", "w"], columns="model", values="WAPE")
for a_, b_ in [("E3a", "E2"), ("E3b", "E2"), ("E3", "E2"), ("E3", "E3a"), ("E3", "E3b")]:
    d = piv[a_] - piv[b_]
    print(f"{a_} − {b_}: ΔWAPE trung bình {d.mean():+.4f}, số lần chạy {a_} tệ hơn: {(d > 0).sum()}/{len(d)}")

## 14. RQ3: quá khớp (train so với validation và test)

train_wape: WAPE trong mẫu của mô hình đã huấn luyện lại, trên các cửa sổ huấn luyện. val_wape: WAPE validation tốt nhất khi early stopping. Khoảng cách = test − train.

In [ ]:
OV = RUNS.merge(MET[["model", "seed", "w", "WAPE"]], on=["model", "seed", "w"]).rename(columns={"WAPE": "test_wape"})
OV["gap_test_train"] = OV.test_wape - OV.train_wape; OV["gap_val_train"] = OV.val_wape - OV.train_wape
OVS = OV.groupby("model")[["train_wape", "val_wape", "test_wape", "gap_val_train", "gap_test_train", "best_epoch"]].agg(["mean", "std"]).reindex(MODELS)
display(OVS.round(4)); OVS.to_csv(f"{OUTPUT_DIR}/p2_overfit.csv")
po = OV.pivot_table(index=["seed", "w"], columns="model", values="gap_test_train")
d = po["E2"] - po["E3b"]
print("Khoảng cách test − train: E2 − E3b trung bình %+.4f; E2 lớn hơn ở %d/%d lần chạy" % (d.mean(), (d > 0).sum(), len(d)))

## 15. Kiểm định thống kê

- Chính: Wilcoxon signed-rank trên MAE theo chuỗi (trung bình qua 5 seed, gộp 5 cửa sổ), n = số chuỗi; hiệu chỉnh Holm cho 5 cặp; độ lớn hiệu ứng: trung vị chênh lệch, % thay đổi MAE trung bình, tương quan rank-biserial.
- Phụ 1: Diebold–Mariano trên chênh lệch sai số tuyệt đối trung bình theo ngày (35 ngày kiểm tra liên tiếp); phương sai HAC Newey–West với độ trễ 6, hiệu chỉnh Harvey–Leybourne–Newbold, phân phối t(n − 1).
- Phụ 2: Wilcoxon trên WAPE theo cặp lần chạy (n = 5 cửa sổ × 5 seed = 25).

Quy ước d = sai số mô hình so sánh − sai số E2; d > 0 nghĩa là E2 tốt hơn.

In [ ]:
PAIRS = [("E2", "E1"), ("E2", "E3"), ("E2", "E3a"), ("E2", "E3b"), ("E2", "E4")]
def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for r, k in enumerate(o): run = max(run, min(1, (len(p) - r) * p[k])); adj[k] = run
    return adj
# sai số tuyệt đối trung bình qua seed cho từng chuỗi × ngày kiểm tra
AE, MK = {}, []
for m in ALL_MODELS:
    AE[m] = np.concatenate([np.mean([np.abs(PRED[(m, sd, w)] - yt(w)[0]) for sd in SEEDS], 0) for w in range(1, K_WINDOWS + 1)], 1)
MK = np.concatenate([yt(w)[1] for w in range(1, K_WINDOWS + 1)], 1)        # N × 35
print("Ma trận sai số:", AE["E2"].shape, "| ô được đánh giá:", int(MK.sum()))
cnt = MK.sum(1); valid = cnt > 0; print("Chuỗi có ít nhất 1 ngày được đánh giá:", int(valid.sum()))
def series_mae(m): return np.where(MK, AE[m], 0).sum(1)[valid] / cnt[valid]
rows = []
for a_, b_ in PAIRS:
    ea, eb = series_mae(a_), series_mae(b_); d = eb - ea
    res = stats.wilcoxon(eb, ea, zero_method="wilcox", alternative="two-sided")
    nz = d[d != 0]; rk = stats.rankdata(np.abs(nz)); rpos, rneg = rk[nz > 0].sum(), rk[nz < 0].sum()
    rows.append({"cặp": f"{a_} vs {b_}", "n": int(len(d)), "MAE_E2": ea.mean(), f"MAE_khác": eb.mean(),
                 "%_thay_đổi_MAE": (ea.mean() / eb.mean() - 1) * 100, "trung_vị_d": float(np.median(d)),
                 "E2_tốt_hơn_%chuỗi": float((d > 0).mean() * 100), "rank_biserial": (rpos - rneg) / (rpos + rneg), "p": res.pvalue})
WIL = pd.DataFrame(rows); WIL["p_Holm"] = holm(WIL.p); WIL["có_ý_nghĩa"] = WIL.p_Holm < ALPHA
display(WIL.round(4)); print(WIL[["cặp", "p", "p_Holm"]].to_string(formatters={"p": "{:.2e}".format, "p_Holm": "{:.2e}".format})); WIL.to_csv(f"{OUTPUT_DIR}/p2_wilcoxon_series.csv", index=False)

In [ ]:
def dm_test(d, h=HORIZON):
    n = len(d); dbar = d.mean(); x = d - dbar; L = h - 1
    g = [np.sum(x[k:] * x[:n - k]) / n for k in range(L + 1)]
    var = g[0] + 2 * sum((1 - k / (L + 1)) * g[k] for k in range(1, L + 1))
    dm = dbar / np.sqrt(var / n)
    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    t = dm * hln; p = 2 * stats.t.sf(abs(t), n - 1)
    return dbar, t, p
rows = []
for a_, b_ in PAIRS:
    la = np.where(MK, AE[a_], 0).sum(0) / MK.sum(0); lb = np.where(MK, AE[b_], 0).sum(0) / MK.sum(0)
    dbar, t, p = dm_test(lb - la)
    rows.append({"cặp": f"{a_} vs {b_}", "n_ngày": len(la), "d_trung_bình": dbar, "DM_HLN": t, "p": p})
DM = pd.DataFrame(rows); DM["p_Holm"] = holm(DM.p)
display(DM.round(4)); print(DM[["cặp", "p", "p_Holm"]].to_string(formatters={"p": "{:.2e}".format, "p_Holm": "{:.2e}".format})); DM.to_csv(f"{OUTPUT_DIR}/p2_dm.csv", index=False)
rows = []
for a_, b_ in PAIRS:
    d = (piv[b_] - piv[a_]).values
    res = stats.wilcoxon(d, zero_method="wilcox")
    rows.append({"cặp": f"{a_} vs {b_}", "n_lần_chạy": len(d), "ΔWAPE_trung_bình": d.mean(), "E2_tốt_hơn": f"{(d > 0).sum()}/{len(d)}", "p": res.pvalue})
WW = pd.DataFrame(rows); WW["p_Holm"] = holm(WW.p)
display(WW.round(4)); print(WW[["cặp", "p", "p_Holm"]].to_string(formatters={"p": "{:.2e}".format, "p_Holm": "{:.2e}".format})); WW.to_csv(f"{OUTPUT_DIR}/p2_wilcoxon_runs.csv", index=False)
mark("tests")

## 16. Biểu đồ

In [ ]:
COL = {"E1": "#85929E", "E2": "#1A5276", "E3": "#B03A2E", "E3a": "#D68910", "E3b": "#239B56", "E4": "#7D3C98"}
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
data = [MET[MET.model == m].WAPE.values for m in ALL_MODELS]
bp = ax[0].boxplot(data, patch_artist=True); ax[0].set_xticks(range(1, len(ALL_MODELS) + 1)); ax[0].set_xticklabels(ALL_MODELS)
for patch, m in zip(bp["boxes"], ALL_MODELS): patch.set_facecolor(COL[m]); patch.set_alpha(0.6)
ax[0].set_ylabel("WAPE"); ax[0].set_title(f"WAPE qua {len(SEEDS)} seed × {K_WINDOWS} cửa sổ")
for m in ALL_MODELS: ax[1].plot(range(1, K_WINDOWS + 1), BYW[m].values, marker="o", color=COL[m], label=m)
ax[1].set_xticks(range(1, K_WINDOWS + 1)); ax[1].set_xticklabels([WIN[w]["test"] for w in range(K_WINDOWS)], rotation=15)
ax[1].set_ylabel("WAPE (trung bình 5 seed)"); ax[1].set_title("WAPE theo tuần kiểm tra")
ax[1].set_ylim(None, BYW.values.max() + 0.02); ax[1].legend(ncol=6, fontsize=9, loc="upper center")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/p2_rolling.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
x = np.arange(4); ms = ["E2", "E3a", "E3b", "E3"]
ax[0].bar(x - 0.2, ABL.loc[ms, "WAPE_promo"], 0.4, label="ngày có khuyến mãi", color="#1A5276")
ax[0].bar(x + 0.2, ABL.loc[ms, "WAPE_nonpromo"], 0.4, label="ngày không khuyến mãi", color="#85929E")
ax[0].set_xticks(x); ax[0].set_xticklabels(ms); ax[0].set_ylim(0.25, ABL[["WAPE_promo", "WAPE_nonpromo"]].values.max() + 0.03)
ax[0].set_ylabel("WAPE"); ax[0].set_title("Ablation: WAPE theo khuyến mãi"); ax[0].legend(ncol=2, loc="upper center")
o = OV.groupby("model")[["train_wape", "val_wape", "test_wape"]].mean().reindex(MODELS)
x = np.arange(len(MODELS))
for k, (c, lab, col) in enumerate([("train_wape", "train", "#AED6F1"), ("val_wape", "validation", "#2874A6"), ("test_wape", "test", "#1A5276")]):
    ax[1].bar(x + (k - 1) * 0.27, o[c], 0.27, label=lab, color=col)
ax[1].set_xticks(x); ax[1].set_xticklabels(MODELS); ax[1].set_ylim(0.2, o.values.max() + 0.04)
ax[1].set_ylabel("WAPE"); ax[1].set_title("Train, validation và test (trung bình 25 lần chạy)"); ax[1].legend(ncol=3, loc="upper center")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/p2_ablation_overfit.png", dpi=150); plt.show()

## 17. Tóm tắt lần chạy

In [ ]:
summary = {"n_series": int(N), "seeds": SEEDS, "k_windows": K_WINDOWS, "n_workers": N_WORKERS, "data_source": SRC,
           "determinism": DET, "times_sec": TIMES, "total_sec": round(time.time() - T0, 1),
           "sum_task_sec": float(RUNS.sec.sum())}
json.dump(summary, open(f"{OUTPUT_DIR}/p2_run_summary.json", "w"), ensure_ascii=False, indent=1)
np.savez_compressed(f"{OUTPUT_DIR}/p2_predictions.npz", **{f"{m}_{s}_{w}": p for (m, s, w), p in PRED.items()})
print(json.dumps(summary, ensure_ascii=False, indent=1))